# Hydrological Drought Propagation Analysis — vectorised implementation

This notebook is an **optimised re-implementation of step 05** (`05_ssi_propagation.ipynb`). It applies exactly the same algorithm, with the same parameters, and writes the same output files, so it can be used as a drop-in replacement for step 05 before running notebooks 06–11.

## What changes and what does not

The algorithm is unchanged: asymmetric search window (`W_DAYS` = 45) → real-overlap filter (`MIN_OVERLAP` = 5) → deterministic four-level tie-break (overlap → severity → temporal proximity → event id) → temporal lag → lag filter (`lag_days ≤ 0`). See `README_propagation.md` and the documentation in `05_ssi_propagation.ipynb`.

Only the computation of the matching step changes:

| | `05_ssi_propagation.ipynb` | this notebook |
|---|---|---|
| Date arithmetic | `pandas.Timestamp` | integer ordinal days (`numpy.int64`) |
| Real overlap | row-wise `DataFrame.apply` | element-wise `numpy` operations |
| Tie-break | `DataFrame.sort_values(...).iloc[0]` | `numpy.lexsort` with the same four keys and directions |

The matching engine is the same function used in the revision analyses (`run_matching_vectorized` in `revision_analyses/propagation_null_core.py`), extended here to also return the overlap length so the full pair table can be rebuilt.

## Validation

Section *Validation against the original implementation* re-runs the original, unvectorised code from notebook 05 on the same inputs and checks that the complete pair table (every row, column, value and dtype) is **identical**. The cell also prints both run times; the speed-up depends on the machine and is of the order of 150× on the Ebro dataset. The comparison can be switched off with `VALIDATE_AGAINST_ORIGINAL = False`.

## Inputs

| File | Description |
|---|---|
| `data/SSI_drought_events.csv` | Event catalogue produced by `04_ssi_drought_events.ipynb` |
| `data/upstream_connectivity.csv` | Per-station list of all hydraulically upstream gauging stations |

## Outputs (same names and contents as step 05)

| File | Description |
|---|---|
| `data/propagation_W45_minov5.csv` | All compatible pairs (before lag filter) |
| `data/propagation_W45_minov5_lagneg.csv` | Lag ≤ 0 pairs with chain metrics (primary pair-level result) |
| `data/chains_summary_W45_minov5.csv` | **Primary output.** One chain per origin event with aggregate metrics |
| `data/Chains_final.csv` | Alias of `chains_summary` used by downstream analysis scripts |


In [ ]:
# ── Imports and algorithm parameters ──────────────────────────────────────────
import time

import numpy as np
import pandas as pd

# ── Parameters (identical to 05_ssi_propagation.ipynb) ────────────────────────
W_DAYS      = 45   # days to expand the origin event window in left side
MIN_OVERLAP = 5    # minimum real overlap days for a candidate to qualify
LAG_MAX     = 0    # retain only pairs where lag_days <= this value

# Re-run the original (slow) implementation and check the results are identical
VALIDATE_AGAINST_ORIGINAL = True

print(f'Parameters: W_DAYS={W_DAYS}, MIN_OVERLAP={MIN_OVERLAP}, LAG_MAX={LAG_MAX}')

In [ ]:
# ── Load drought events catalogue ─────────────────────────────────────────────
events = pd.read_csv(
    'data/SSI_drought_events.csv',
    parse_dates=['start_date', 'end_date', 'peak_date']
)
events = events.sort_values(['station_id', 'event_id']).reset_index(drop=True)

print(f'Events loaded : {len(events):,} events across {events["station_id"].nunique()} stations')
print(f'Date range    : {events["start_date"].min().date()} → {events["end_date"].max().date()}')

In [ ]:
# ── Load upstream connectivity ─────────────────────────────────────────────────
conn_raw = pd.read_csv('data/upstream_connectivity.csv', sep=';', dtype=str)
conn_raw.columns = conn_raw.columns.str.strip()
conn_raw['station_id']     = conn_raw['station_id'].str.strip().astype(int)
conn_raw['upstream_chain'] = conn_raw['upstream_chain'].str.strip()

def parse_chain(s):
    '''Parse a comma-separated string of station IDs into a list of ints.'''
    if pd.isna(s) or s == '':
        return []
    return [int(x.strip()) for x in s.split(',') if x.strip()]

connectivity = {
    row['station_id']: parse_chain(row['upstream_chain'])
    for _, row in conn_raw.iterrows()
}

n_with_upstream = sum(1 for v in connectivity.values() if v)
print(f'Stations in connectivity file : {len(connectivity)}')
print(f'  With upstream stations      : {n_with_upstream}')
print(f'  Headwaters (no upstream)    : {len(connectivity) - n_with_upstream}')

## Vectorised matching

Each station's catalogue is converted once into `numpy` arrays, with dates stored as integer ordinal days. For every origin event and upstream station, the search-window filter, the real-overlap computation and the tie-break then operate on whole arrays:

- **Search window:** `end_u ≥ start_o − W_DAYS` and `start_u ≤ start_o`.
- **Real overlap:** `max(0, min(end_o, end_u) − max(start_o, start_u) + 1)`, the same inclusive count as `real_overlap()` in notebook 05.
- **Tie-break:** `numpy.lexsort` evaluates its *last* key first, so the keys are passed as `(event_id, |start difference|, −severity, −overlap)`. This reproduces the order overlap desc → severity desc → |start difference| asc → event_id asc.

As in notebook 05, the best candidate is chosen **before** the lag filter is applied.

In [ ]:
# ── Vectorised matching engine ─────────────────────────────────────────────────

def build_station_arrays(events_df):
    '''station_id -> dict of numpy arrays (ordinal-day ints, severity, event_id).'''
    out = {}
    for sid, grp in events_df.groupby('station_id'):
        grp = grp.sort_values('event_id')
        out[sid] = dict(
            start=grp['start_date'].values.astype('datetime64[D]').astype(np.int64),
            end=grp['end_date'].values.astype('datetime64[D]').astype(np.int64),
            sev=grp['severity'].values.astype(np.float64),
            eid=grp['event_id'].values.astype(np.int64),
        )
    return out


def run_matching_vectorized(station_arr, connectivity):
    '''Best upstream candidate per (origin event, upstream station), before the lag filter.

    Same logic as run_matching_vectorized() in revision_analyses/propagation_null_core.py;
    additionally returns the overlap length.
    Returns tuples: (origin_station, origin_event_id, upstream_station,
                     upstream_event_id, overlap_days, lag_days).
    '''
    origin_stations = [sid for sid, chain in connectivity.items()
                       if chain and sid in station_arr]
    pairs = []
    for origin_sid in origin_stations:
        upstream_sids = [s for s in connectivity[origin_sid] if s in station_arr]
        o = station_arr[origin_sid]
        for i in range(len(o['start'])):
            s_o, e_o = o['start'][i], o['end'][i]
            w_start, w_end = s_o - W_DAYS, s_o
            for up_sid in upstream_sids:
                u = station_arr[up_sid]
                mask = (u['end'] >= w_start) & (u['start'] <= w_end)
                if not mask.any():
                    continue
                s2, e2, sv2, id2 = u['start'][mask], u['end'][mask], u['sev'][mask], u['eid'][mask]
                overlap = np.maximum(np.minimum(e_o, e2) - np.maximum(s_o, s2) + 1, 0)
                keep = overlap >= MIN_OVERLAP
                if not keep.any():
                    continue
                ov, s3, sv3, id3 = overlap[keep], s2[keep], sv2[keep], id2[keep]
                absdiff = np.abs(s3 - s_o)
                b = np.lexsort((id3, absdiff, -sv3, -ov))[0]
                pairs.append((origin_sid, int(o['eid'][i]), up_sid, int(id3[b]),
                              int(ov[b]), int(s3[b] - s_o)))
    return pairs


print('Vectorised matching engine defined.')

In [ ]:
# ── Run the matching and rebuild the full pair table ───────────────────────────
station_arr = build_station_arrays(events)

t0 = time.perf_counter()
pairs = run_matching_vectorized(station_arr, connectivity)
t_vectorised = time.perf_counter() - t0

pairs_df = pd.DataFrame(pairs, columns=['origin_station', 'origin_event_id', 'upstream_station',
                                        'upstream_event_id', 'overlap_days', 'lag_days'])

# Attach event attributes from the catalogue (origin and upstream side)
ev = events.set_index(['station_id', 'event_id'])[['start_date', 'end_date', 'duration', 'severity']]
orig_attr = ev.loc[list(zip(pairs_df['origin_station'], pairs_df['origin_event_id']))].reset_index(drop=True)
up_attr   = ev.loc[list(zip(pairs_df['upstream_station'], pairs_df['upstream_event_id']))].reset_index(drop=True)

prop_full = pd.DataFrame({
    'origin_station'   : pairs_df['origin_station'],
    'origin_event_id'  : pairs_df['origin_event_id'],
    'origin_start'     : orig_attr['start_date'],
    'origin_end'       : orig_attr['end_date'],
    'origin_duration'  : orig_attr['duration'],
    'origin_severity'  : orig_attr['severity'],
    'upstream_station' : pairs_df['upstream_station'],
    'upstream_event_id': pairs_df['upstream_event_id'],
    'upstream_start'   : up_attr['start_date'],
    'upstream_end'     : up_attr['end_date'],
    'upstream_duration': up_attr['duration'],
    'overlap_days'     : pairs_df['overlap_days'],
    'lag_days'         : pairs_df['lag_days'],
    'upstream_severity': up_attr['severity'],
})
prop_full = prop_full.sort_values(
    ['origin_station', 'origin_event_id', 'lag_days', 'upstream_station']
).reset_index(drop=True)

print(f'Vectorised matching time             : {t_vectorised:.3f} s')
print(f'Compatible pairs (before lag filter) : {len(prop_full):,}')
print(f'Origin events matched to ≥1 upstream : '
      f'{prop_full.groupby(["origin_station","origin_event_id"]).ngroups:,}')
prop_full.head(10)

## Validation against the original implementation

The next cell contains the matching loop of `05_ssi_propagation.ipynb` (without the progress bar) and runs it on the same inputs. The two pair tables must be identical: same rows in the same order, same columns, same values and same dtypes (`pandas.testing.assert_frame_equal`). If they differ, the cell raises an error and nothing is saved.

In [ ]:
# ── Original implementation (from 05_ssi_propagation.ipynb) ────────────────────

def real_overlap(s_o, e_o, s_u, e_u):
    delta = (min(e_o, e_u) - max(s_o, s_u)).days + 1
    return max(0, delta)


def select_best_candidate(candidates):
    return (
        candidates
        .sort_values(['overlap_days', 'severity', 'abs_start_diff', 'event_id'],
                     ascending=[False, False, True, True])
        .iloc[0]
    )


def run_matching_original(events, connectivity):
    station_events = {sid: grp.reset_index(drop=True) for sid, grp in events.groupby('station_id')}
    W = pd.Timedelta(days=W_DAYS)
    origin_stations = [sid for sid, chain in connectivity.items()
                       if chain and sid in station_events]
    all_pairs = []
    for origin_sid in origin_stations:
        origin_evts = station_events[origin_sid]
        upstream_with_events = [s for s in connectivity[origin_sid] if s in station_events]
        for _, orig in origin_evts.iterrows():
            s_o, e_o = orig['start_date'], orig['end_date']
            w_start, w_end = s_o - W, s_o
            for up_sid in upstream_with_events:
                up_evts = station_events[up_sid]
                mask = (up_evts['end_date'] >= w_start) & (up_evts['start_date'] <= w_end)
                candidates = up_evts[mask].copy()
                if candidates.empty:
                    continue
                candidates['overlap_days'] = candidates.apply(
                    lambda r: real_overlap(s_o, e_o, r['start_date'], r['end_date']), axis=1)
                candidates = candidates[candidates['overlap_days'] >= MIN_OVERLAP]
                if candidates.empty:
                    continue
                candidates['abs_start_diff'] = (candidates['start_date'] - s_o).abs().dt.days
                best = select_best_candidate(candidates)
                all_pairs.append({
                    'origin_station'   : origin_sid,
                    'origin_event_id'  : orig['event_id'],
                    'origin_start'     : s_o,
                    'origin_end'       : e_o,
                    'origin_duration'  : orig['duration'],
                    'origin_severity'  : orig['severity'],
                    'upstream_station' : up_sid,
                    'upstream_event_id': best['event_id'],
                    'upstream_start'   : best['start_date'],
                    'upstream_end'     : best['end_date'],
                    'upstream_duration': best['duration'],
                    'overlap_days'     : int(best['overlap_days']),
                    'lag_days'         : (best['start_date'] - s_o).days,
                    'upstream_severity': best['severity'],
                })
    return (pd.DataFrame(all_pairs)
            .sort_values(['origin_station', 'origin_event_id', 'lag_days', 'upstream_station'])
            .reset_index(drop=True))


if VALIDATE_AGAINST_ORIGINAL:
    t0 = time.perf_counter()
    prop_full_original = run_matching_original(events, connectivity)
    t_original = time.perf_counter() - t0

    pd.testing.assert_frame_equal(prop_full, prop_full_original)

    print(f'Original implementation   : {t_original:6.2f} s -> {len(prop_full_original):,} pairs')
    print(f'Vectorised implementation : {t_vectorised:6.3f} s -> {len(prop_full):,} pairs')
    print(f'Speed-up                  : {t_original / t_vectorised:.0f}x')
    print('Pair tables are IDENTICAL (rows, order, columns, values and dtypes).')
else:
    print('Validation skipped (VALIDATE_AGAINST_ORIGINAL = False).')

## Post-processing: lag filter, chain metrics and outputs

From here on the code is the same as in `05_ssi_propagation.ipynb`; see that notebook for the definitions of `chain_size`, `n_upstream_with_events` and `propagation_fraction`, and for the rationale of computing one chain per origin event with no post-hoc merging.

In [ ]:
# ── Lag filter ─────────────────────────────────────────────────────────────────
prop_neg = prop_full[prop_full['lag_days'] <= LAG_MAX].copy().reset_index(drop=True)

print(f'Pairs after lag filter (lag ≤ {LAG_MAX}): {len(prop_neg):,}')
print(f'Pairs excluded (positive lag)           : {len(prop_full) - len(prop_neg):,}')
print()

# ── Chain size per origin event ────────────────────────────────────────────────
chain_size = (
    prop_neg
    .groupby(['origin_station', 'origin_event_id'])['upstream_station']
    .nunique()
    .rename('chain_size')
    .reset_index()
)

# ── Denominator: upstream stations present in the event catalogue ──────────────
upstream_available = {
    sid: sum(1 for s in chain if s in station_arr)
    for sid, chain in connectivity.items()
}
chain_size['n_upstream_with_events'] = chain_size['origin_station'].map(upstream_available)
chain_size['propagation_fraction']   = (
    chain_size['chain_size'] / chain_size['n_upstream_with_events']
).round(4)

prop_neg = prop_neg.merge(chain_size, on=['origin_station', 'origin_event_id'], how='left')

print('Chain size summary (across all origin events):')
print(chain_size['chain_size'].describe().round(2))
print()
print('Propagation fraction summary:')
print(chain_size['propagation_fraction'].describe().round(3))

In [ ]:
# ── Sanity checks ──────────────────────────────────────────────────────────────
assert (prop_neg['lag_days'] <= 0).all(), 'Lag filter failed: positive lags present'
assert (prop_neg['overlap_days'] >= MIN_OVERLAP).all(), 'Overlap threshold violated'
assert prop_neg['propagation_fraction'].between(0, 1).all(), 'Propagation fraction out of [0,1]'
dups = prop_neg.duplicated(['origin_station', 'origin_event_id', 'upstream_station'])
assert not dups.any(), f'{dups.sum()} duplicate (origin, event, upstream) triples found'

print('All sanity checks passed.')
print()

print('Lag distribution (days):')
bins = [(-9999, -90), (-90, -30), (-30, -7), (-7, 0)]
for lo, hi in bins:
    n = ((prop_neg['lag_days'] > lo) & (prop_neg['lag_days'] <= hi)).sum()
    pct = n / len(prop_neg) * 100
    label = f'({lo}, {hi}]' if lo != -9999 else f'< {hi}'
    print(f'  {label:>15} days: {n:>5} pairs ({pct:.1f}%)')

print()
print(f'Median lag : {prop_neg["lag_days"].median():.0f} days')
print(f'Mean lag   : {prop_neg["lag_days"].mean():.1f} days')

In [ ]:
# ── Save pair-level outputs ────────────────────────────────────────────────────
origin_stations = [sid for sid, chain in connectivity.items() if chain and sid in station_arr]

out_full = f'data/propagation_W{W_DAYS}_minov{MIN_OVERLAP}.csv'
prop_full.to_csv(out_full, index=False)
print(f'Saved {len(prop_full):,} pairs → {out_full}')

out_neg  = f'data/propagation_W{W_DAYS}_minov{MIN_OVERLAP}_lagneg.csv'
prop_neg.to_csv(out_neg, index=False)
print(f'Saved {len(prop_neg):,} pairs → {out_neg}')

print()
print('=== Final summary ===')
print(f'Origin stations processed              : {len(origin_stations)}')
print(f'Total origin events                    : '
      f'{sum(len(station_arr[s]["eid"]) for s in origin_stations):,}')
print(f'Compatible pairs (lag ≤ {LAG_MAX})         : {len(prop_neg):,}')
print(f'Origin events with ≥1 upstream match   : '
      f'{prop_neg.groupby(["origin_station","origin_event_id"]).ngroups:,}')
print(f'Mean chain size                        : '
      f'{chain_size["chain_size"].mean():.2f}')
print(f'Mean propagation fraction              : '
      f'{chain_size["propagation_fraction"].mean():.3f}')

In [ ]:
# ── Chain summary: PRIMARY analytical output (one chain per origin event) ──────
prop_neg['chain_id'] = (
    prop_neg['origin_station'].astype(str) + '_' +
    prop_neg['origin_event_id'].astype(str)
)

hm3 = events[['station_id', 'event_id', 'severity_hm3']]

prop_hm3 = prop_neg.merge(
    hm3.rename(columns={'station_id'  : 'upstream_station',
                        'event_id'    : 'upstream_event_id',
                        'severity_hm3': 'upstream_severity_hm3'}),
    on=['upstream_station', 'upstream_event_id'], how='left'
)

origin_fields = (
    prop_neg.drop_duplicates('chain_id')
    [['chain_id', 'origin_station', 'origin_event_id',
      'origin_start', 'origin_end', 'origin_severity',
      'chain_size', 'n_upstream_with_events', 'propagation_fraction']]
    .merge(
        hm3.rename(columns={'station_id'  : 'origin_station',
                            'event_id'    : 'origin_event_id',
                            'severity_hm3': 'origin_severity_hm3'}),
        on=['origin_station', 'origin_event_id'], how='left'
    )
)

upstream_agg = (
    prop_hm3.groupby('chain_id')
    .agg(
        chain_start       = ('upstream_start',        'min'),
        upstream_end_max  = ('upstream_end',           'max'),
        lag_mean          = ('lag_days',               'mean'),
        upstream_sev_sum  = ('upstream_severity',      'sum'),
        upstream_hm3_sum  = ('upstream_severity_hm3',  'sum'),
    )
    .reset_index()
)

chains = origin_fields.merge(upstream_agg, on='chain_id')

chains['chain_end']      = chains[['upstream_end_max', 'origin_end']].max(axis=1)
chains['chain_duration'] = (chains['chain_end'] - chains['chain_start']).dt.days + 1
chains['chain_size']     = chains['chain_size'] + 1
chains['chain_severity']     = (chains['upstream_sev_sum'] + chains['origin_severity']).round(4)
chains['chain_severity_hm3'] = (chains['upstream_hm3_sum'] + chains['origin_severity_hm3']).round(4)
chains['lag_mean']           = chains['lag_mean'].round(1)

chains = (
    chains[[
        'chain_id', 'origin_station',
        'origin_start', 'origin_end',
        'chain_start', 'chain_end', 'chain_duration',
        'chain_size', 'propagation_fraction',
        'lag_mean',
        'chain_severity', 'chain_severity_hm3',
    ]]
    .sort_values(['origin_station', 'chain_start'])
    .reset_index(drop=True)
)

chains.to_csv(f'data/chains_summary_W{W_DAYS}_minov{MIN_OVERLAP}.csv', index=False)
chains.to_csv('data/Chains_final.csv', index=False)  # alias used by plot/table scripts

print(f'Chain summary: {len(chains):,} chains across {chains["origin_station"].nunique()} origin stations')
print()
chains.head(10)

In [ ]:
# Primary output summary statistics
print(f'Primary output: {len(chains):,} chains (one per origin event)')
chains.describe()